# 카페 매출 예측을 위한 전처리 파이프라인

가상의 매장 매출 데이터 `cafe_sales.csv`를 **예측 모델에 넣을 수 있는 상태**까지 정제합니다.

흐름: 데이터 로드 → **탐색** → **결측치** → **이상치** → **중복** → **범주형 인코딩** → **피처 벡터 생성**

| 컬럼 | 타입 | 설명 |
|---|---|---|
| store_id | string | 매장 코드 |
| date | string | 매출 일자 (YYYY-MM-DD) |
| weather | string | 날씨 (sunny/cloudy/rain/snow) — **일부 결측** |
| temp | double | 평균 기온(°C) — **이상치 포함** |
| weekday | string | 요일 (Mon~Sun) |
| customer_cnt | int | 방문 고객 수 |
| sales | double | 일 매출 — **결측·이상치 포함** |

In [ ]:
# Step 0-1. SparkSession 생성
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("CafeSalesPreprocessing")
    .master("local[*]")
    .getOrCreate()
)

# Step 0-2. 실습 데이터 로드
df = spark.read.csv("data/cafe_sales.csv", header=True, inferSchema=True)

print(f"전체 행 수: {df.count():,}")
df.printSchema()
df.show(5)

## Step 1. 데이터 탐색 — 정제 전에 문제를 먼저 찾는다

In [ ]:
# 1-1) 수치형 컬럼 기초 통계 — min/max에서 이상한 값이 보이는가?
df.select("temp", "customer_cnt", "sales").describe().show()

In [ ]:
# 1-2) 컬럼별 결측 건수 확인
df.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in df.columns
]).show()

In [ ]:
# 1-3) 범주형 컬럼의 값 분포 — 예상 밖의 범주가 있는가?
df.groupBy("weather").count().orderBy(F.desc("count")).show()
df.groupBy("weekday").count().orderBy(F.desc("count")).show()

**탐색에서 발견한 문제를 적어보세요** (아래 정제 단계의 근거가 됩니다)
- sales 최댓값이 9,900만? 음수 매출? → 이상치
- weather/sales에 결측 → 결측치 처리 필요
- temp 최댓값 250도 → 입력 실수 가능성

## Step 2. 결측치 처리 — 컬럼 성격에 따라 전략이 다르다

In [ ]:
# 2-1) weather 결측 → 'Unknown' 범주로 대체 (범주형: 별도 범주가 안전)
df1 = df.fillna({"weather": "Unknown"})

# 2-2) sales 결측 → 행 삭제
#      매출은 모델이 맞혀야 할 '정답(라벨)'이므로 평균으로 채우면
#      인위적인 학습 데이터가 됩니다. 정답이 없는 행은 학습에서 제외!
before = df1.count()
df1 = df1.dropna(subset=["sales"])
print(f"sales 결측 제거: {before:,} → {df1.count():,} 행")

## Step 3. 이상치 처리 — 통계 기준(IQR) + 도메인 기준

In [ ]:
# 3-1) 도메인 기준: 매출은 0 이상, 기온은 -30~45도
df2 = df1.filter(F.col("sales") >= 0).filter(F.col("temp").between(-30, 45))

# 3-2) 통계 기준: sales에 IQR 적용
# IQR(사분위 범위) = Q3-Q1, "데이터 가운데 50%가 퍼진 정도"입니다.
# 1.5배는 통계학에서 관례적으로 쓰는 계수로, 이 범위를 벗어나면 극단치로 봅니다.
q1, q3 = df2.approxQuantile("sales", [0.25, 0.75], 0.0)
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"IQR 허용 범위: {low:,.0f} ~ {high:,.0f}")

before = df2.count()
df2 = df2.filter(F.col("sales").between(low, high))
print(f"이상치 제거: {before:,} → {df2.count():,} 행")
df2.select("sales").describe().show()  # max가 정상 범위로 돌아왔는지 확인

## Step 4. 중복 제거 — store_id + date가 기본 키

In [ ]:
before = df2.count()
df3 = df2.dropDuplicates(["store_id", "date"])
print(f"중복 제거: {before:,} → {df3.count():,} 행")

## Step 5. 범주형 인코딩 — 문자를 모델이 읽을 수 있는 숫자로

In [ ]:
# StringIndexer: 범주 → 정수 인덱스, OneHotEncoder: 인덱스 → 원-핫 벡터
#
# 왜 정수 인덱스만으로 끝내지 않고 원-핫까지 가는가?
# StringIndexer만 쓰면 weather가 sunny=0, rain=1, snow=2처럼 숫자로 바뀌는데,
# 회귀/거리 기반 모델은 이걸 "snow가 sunny보다 2배 크다"는 크기 관계로
# 잘못 해석할 수 있습니다. weather는 순서가 없는 범주라서, 원-핫 인코딩으로
# 각 범주를 독립된 0/1 컬럼으로 풀어줘야 이런 왜곡이 생기지 않습니다.
from pyspark.ml.feature import StringIndexer, OneHotEncoder

indexer = StringIndexer(
    inputCols=["weather", "weekday"],
    outputCols=["weather_idx", "weekday_idx"],
)
df4 = indexer.fit(df3).transform(df3)

encoder = OneHotEncoder(
    inputCols=["weather_idx", "weekday_idx"],
    outputCols=["weather_vec", "weekday_vec"],
)
df4 = encoder.fit(df4).transform(df4)

df4.select("weather", "weather_idx", "weather_vec",
           "weekday", "weekday_idx", "weekday_vec").show(5, truncate=False)

## Step 6. 피처 벡터 생성 — 모델 입력 완성

In [ ]:
from pyspark.ml.feature import VectorAssembler

# VectorAssembler는 서로 다른 컬럼들(숫자형 temp/customer_cnt +
# 원-핫 벡터인 weather_vec/weekday_vec)을 하나의 벡터로 합쳐줍니다.
# 스파크 ML의 모든 모델은 입력을 "features"라는 이름의 벡터 컬럼 하나로
# 받기 때문에, 학습 직전에 항상 이 단계를 거칩니다.
assembler = VectorAssembler(
    inputCols=["temp", "customer_cnt", "weather_vec", "weekday_vec"],
    outputCol="features",
)
df_final = assembler.transform(df4).select(
    "store_id", "date", "features", F.col("sales").alias("label"))

df_final.show(5, truncate=False)
print("전처리 파이프라인 완성 — 이 features/label이 예측 모델의 입력입니다.")

In [ ]:
# (보너스) 정말 예측이 되는지 미리 맛보기 — 다음 회차 예고
from pyspark.ml.regression import LinearRegression

train, test = df_final.randomSplit([0.8, 0.2], seed=42)
model = LinearRegression(featuresCol="features", labelCol="label").fit(train)
print(f"테스트 R² = {model.evaluate(test).r2:.3f}")

### 생각해 볼 질문
1. weather 결측은 'Unknown'으로 **채우고**, sales 결측은 **삭제**했습니다. 왜 전략이 달라야 할까요?
2. 요일(weekday)을 원-핫 대신 Mon=1, Tue=2, ... 숫자로 그대로 쓰면 어떤 문제가 생길까요?
3. IQR로 지운 9,900만 매출이 만약 대규모 단체 주문이었다면, 지우는 게 맞았을까요?
4. store_id + date 조합을 기본 키로 본 이유는 무엇일까요?

In [ ]:
spark.stop()